In [1]:
import sys
!{sys.executable} -m pip install nltk


[notice] A new release of pip is available: 25.3 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import hashlib
import json
import os
import re
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score

# Dodanie katalogu głównego do ścieżki Pythona, aby zaimportować klasy z features/
sys.path.append(str(Path.cwd().parent))

from features.brand_imputer import BrandImputer
from features.category_probability_extractor import CategoryProbabilityExtractor
from features.pack_size_probability_extractor import (
    PackSizeProbabilityExtractor,
)

SEED = 42
np.random.seed(SEED)

In [3]:
DATA_DIR = Path('../data')

train = pd.read_csv(DATA_DIR / 'train.csv')
test = pd.read_csv(DATA_DIR / 'test.csv')
sample_sub = pd.read_csv(DATA_DIR / 'sample_submission.csv')


def seller_folds(sellers, n_splits=5, seed=SEED):
  assignment = np.array([
      int(hashlib.sha256(f'{seed}:{s}'.encode('utf-8')).hexdigest()[:16], 16)
      % n_splits
      for s in sellers.astype(str)
  ])
  for fold in range(n_splits):
    fit = np.flatnonzero(assignment != fold)
    valid = np.flatnonzero(assignment == fold)
    yield fit, valid


# Weryfikacja rozłączności sprzedawców
assert set(train.seller_id).isdisjoint(set(test.seller_id))
print(f'Train shape: {train.shape}, Test shape: {test.shape}')
print(f'Odsetek błędów (target=1): {train.target.mean():.4f}')

Train shape: (16000, 11), Test shape: (10000, 10)
Odsetek błędów (target=1): 0.1701


In [4]:
def compute_price_weight_features(df):
  x = df.copy()

  effective_pack = x['predicted_pack_size'].fillna(x['pack_size']).fillna(1.0)
  effective_pack = effective_pack.clip(lower=1.0)

  # Ceny i wagi jednostkowe
  x['unit_price'] = x['price_pln'] / effective_pack
  x['unit_weight'] = x['shipping_weight_kg'] / effective_pack

  x['log_unit_price'] = np.log1p(x['unit_price'].clip(lower=0))
  x['log_unit_weight'] = np.log1p(x['unit_weight'].clip(lower=0))
  x['log_price'] = np.log1p(x['price_pln'].clip(lower=0))
  x['log_weight'] = np.log1p(x['shipping_weight_kg'].clip(lower=0))

  # Flagi niezgodności
  x['category_mismatch'] = (
      (x['declared_category'] != x['predicted_category'])
      & x['declared_category'].notna()
  ).astype(int)
  x['pack_size_mismatch'] = (
      (x['pack_size'] != x['predicted_pack_size']) & x['pack_size'].notna()
  ).astype(int)

  x['cat_prob_delta'] = 1.0 - x['declared_cat_prob'].clip(lower=0.0)
  x['pack_prob_delta'] = 1.0 - x['declared_pack_size_prob'].clip(lower=0.0)

  # Długości tekstu i brakujące wartości
  x['title_len'] = x['title'].fillna('').astype(str).str.len()
  x['desc_len'] = x['description'].fillna('').astype(str).str.len()
  x['has_missing_price'] = x['price_pln'].isna().astype(int)
  x['has_missing_weight'] = x['shipping_weight_kg'].isna().astype(int)

  # Z-score cen i wag per kategoria
  cat_col = x['predicted_category'].fillna('missing')
  price_stats = x.groupby(cat_col)['log_unit_price'].agg(['mean', 'std'])
  weight_stats = x.groupby(cat_col)['log_unit_weight'].agg(['mean', 'std'])

  x['z_score_price'] = (
      x['log_unit_price'] - cat_col.map(price_stats['mean'])
  ) / (cat_col.map(price_stats['std']) + 1e-5)
  x['z_score_price_abs'] = x['z_score_price'].abs()

  x['z_score_weight'] = (
      x['log_unit_weight'] - cat_col.map(weight_stats['mean'])
  ) / (cat_col.map(weight_stats['std']) + 1e-5)
  x['z_score_weight_abs'] = x['z_score_weight'].abs()

  return x

